# 1. Road accidents data preparation

In [1]:
import os

In [2]:
os.environ['PROJ_DATA'] = r'C:\Users\batur\micromamba\envs\geopython2025\Library\share\proj'

In [3]:
os.environ['PROJ_LIB'] = r'C:\Users\batur\micromamba\envs\geopython2025\Library\share\proj'

First, it needs to download the accident data from the Estonian Road Administration website. The data in .csv format is available at eesti.ee, via the link provided in the data section of the README. The table provides a detailed account of every accident in Estonia that has occurred since 2011. Important! The data is in Estonian, so further explanations in English are required.

In [4]:
import pandas as pd

In [5]:
from IPython.display import display
import ipywidgets as widgets

In [6]:
# load the accidents data
upload = widgets.FileUpload( accept='lo_2011_2026.csv', multiple=False )

csv_delimiter = ";"
text_encoding = "utf8"

In [7]:
lo1125 = pd.read_csv('lo_2011_2026.csv', encoding=text_encoding, sep=csv_delimiter)

In [8]:
#lo1125.head(10) private data

Too more colums to represent all data in pandas, might be good to see a list of columns, check them in Excel and select nessesary columns. 

In [9]:
for col in lo1125.columns:
    print(col)

Juhtumi nr
Toimumisaeg
Isikuid
Hukkunuid
Sõidukeid
Vigastatuid
Aadress
Tänav
Maja nr
Ristuv tänav
Tee nr
Tee km
Maakond
Omavalitsus
Asustusüksus
Asula
Liiklusõnnetuse liik
Liiklusõnnetuse liik (detailne)
Kergliikurijuhi osalusel
Jalakäija osalusel
Kaassõitja osalusel
Maastikusõiduki juhi osalusel
Eaka (65+) mootorsõidukijuhi osalusel
Bussijuhi osalusel
Veoautojuhi osalusel
Ühissõidukijuhi osalusel
Sõiduautojuhi osalusel
Mootorratturi osalusel
Mopeedijuhi osalusel
Jalgratturi osalusel
Alaealise osalusel
Esmase juhiloa omaniku osalusel
Turvavarustust mitte kasutanud isiku osalusel
Mootorsõidukijuhi osalusel
Tüüpskeemi nr
Tüüpskeem
Tee tüüp
Tee tüüp (detailne)
Tee liik
Tee element
Tee element (detailne)
Tee objekt
Kurvilisus
Tee tasasus
Tee seisund
Teekate
Teekatte seisund
Sõiduradade arv
Lubatud sõidukiirus
Ilmastik
Valgustus
Valgustus (detailne)
X koordinaat
Y koordinaat


For analysis we need only road accidents in Tallinn on 2021-2024. For that we need column "Omavalitsus" (municipality).

P.S. We exclude 2020 due to pandemic limitations of movements.

In [10]:
# Filter only rows related to Tallinn
lo_tallinn = lo1125[lo1125["Omavalitsus"] == "Tallinn"]

In [11]:
# Filter only 2021-2024
# Assumption: "Toimumusaeg" (Event time) contains a date as text, e.g. "2024-06-15"
lo_tal_21_24 = lo_tallinn[
    (lo_tallinn["Toimumisaeg"].astype(str).str[0:4] >= "2021") &
    (lo_tallinn["Toimumisaeg"].astype(str).str[0:4] <= "2024")
]

Now it needs to save the data in Excel to view the entire table.

In [12]:
# Save directly to Excel format
lo_tal_21_24.to_excel('road_accs_tln_21_24.xlsx', 
                index=False,          
                engine='openpyxl')     # engine for working with xlsx
# Check in Excel and selest columns that is nessesary for analysis.

Check in Excel and selest columns that is nessesary for analysis.

After watching data in MS Excel it was understood, that next columns might be useful for analysis: Juhtumi nr, Toimumisaeg, Hukkunuid, Tänav, Ristuv tänav, Asustusüksus, Liiklusõnnetuse liik (detailne), Jalakäija osalusel, Tee tüüp (detailne), Teekatte seisund, Lubatud sõidukiirus, Ilmastik, Valgustus, X koordinaat, Y koordinaat. In English that columns mean: Incident number, Time of occurrence, Fatalities, Street, Intersection, Settlement unit, Type of traffic accident, Pedestrian involved, Road type (detailed), Road surface condition, Speed limit, Weather, Lighting, X coordinate, Y coordinate.

In [13]:
# List of required columns
selected_columns = [
    "Juhtumi nr", # Incident number
    "Toimumisaeg", # Time of occurrence
    "Hukkunuid", # Fatalities
    "Tänav", # Street
    "Ristuv tänav", # Intersection
    "Asustusüksus", # Settlement unit
    "Liiklusõnnetuse liik (detailne)", # Type of traffic accident (detail)
    "Jalakäija osalusel", # Pedestrian involved
    "Tee tüüp (detailne)", # Road type (detailed)
    "Teekatte seisund", # Road surface condition
    "Lubatud sõidukiirus", # Speed limit
    "Ilmastik", # Weather
    "Valgustus", # Lighting
    "X koordinaat", # X coordinate 
    "Y koordinaat" # Y coordinate
]

# Select only these columns
lo_tal_21_24_selected = lo_tal_21_24[selected_columns]

# Check a result
print(f"Filtered data size: {lo_tal_21_24_selected.shape}")
print(f"Number of rows: {lo_tal_21_24_selected.shape[0]}")
print(f"Number of columns: {lo_tal_21_24_selected.shape[1]}")
#lo_tal_2024_selected.head(10)

Filtered data size: (2759, 15)
Number of rows: 2759
Number of columns: 15


Here it is necessary to delete those rows where coordinates are missing, since the analysis requires the location of the accident.

In [14]:
# We delete lines where the X or Y coordinate is missing
lo_tal_21_24_clean = lo_tal_21_24_selected.dropna(subset=["X koordinaat", "Y koordinaat"])

print(f"Were rows: {len(lo_tal_21_24_selected)}")
print(f"Become rows: {len(lo_tal_21_24_clean)}")
print(f"Deleted rows: {len(lo_tal_21_24_selected) - len(lo_tal_21_24_clean)}")

# Let's see how much is left
print(f"\nRoad accidents with geometry: {len(lo_tal_21_24_clean)}")

Were rows: 2759
Become rows: 2755
Deleted rows: 4

Road accidents with geometry: 2755


For easier analysis in English, the columns will be renamed into English.

In [15]:
rename_columns = {
    "Juhtumi nr": "case_id",
    "Toimumisaeg": "date_time",
    "Hukkunuid": "fatalities",
    "Vigastatuid": "injuries",
    "Tänav": "street",
    "Ristuv tänav": "ins_street",
    "Asustusüksus": "settl_unit",
    "Liiklusõnnetuse liik (detailne)": "accs_type",
    "Jalakäija osalusel": "pedestrian",
    "Tee tüüp (detailne)": "road_type",
    "Teekatte seisund": "road_cond",
    "Lubatud sõidukiirus": "speed_kmh",
    "Ilmastik": "weather",
    "Valgustus": "lighting",
    "X koordinaat": "x_coord",
    "Y koordinaat": "y_coord"
}

# Apply renaming
lo_tal_21_24_clean = lo_tal_21_24_clean.rename(columns=rename_columns)

print("Новые названия колонок:")
print(lo_tal_21_24_clean.columns.tolist())

# Let's look at the first lines with new names
lo_tal_21_24_clean.head(10)

Новые названия колонок:
['case_id', 'date_time', 'fatalities', 'street', 'ins_street', 'settl_unit', 'accs_type', 'pedestrian', 'road_type', 'road_cond', 'speed_kmh', 'weather', 'lighting', 'x_coord', 'y_coord']


,case_id,date_time,fatalities,street,ins_street,settl_unit,accs_type,pedestrian,road_type,road_cond,speed_kmh,weather,lighting,x_coord,y_coord
4,2101230012484,2023-04-14 12:23:00,0,Peterburi tee,Kuuli tn,Lasnamäe linnaosa,Sõidukite külgkokkupõrge,0.0,Tänav,Kuiv,50.0,Selged olud,Valge aeg,6588729.0,548805.0
11,3100220503923,2022-08-22 19:21:00,0,Reidi tee,Narva mnt,Kesklinna linnaosa,Sõidukite külgkokkupõrge,0.0,Jalg- ja jalgrattatee,Kuiv,50.0,Selged olud,Valge aeg,6589755.0,544773.0
14,3100220351609,2022-06-23 19:28:00,0,Mustakivi tee,NaN,Lasnamäe linnaosa,Sõiduki ümberpaiskumine teel,0.0,Tänav,Kuiv,50.0,Selged olud,Valge aeg,6589532.0,548925.0
15,2101230085773,2023-07-23 12:55:00,0,Järvevana tee,NaN,Kesklinna linnaosa,Sõiduki ümberpaiskumine teel,0.0,Jalg- ja jalgrattatee,Kuiv,NaN,Pilvised olud,Valge aeg,6585478.0,542033.0
29,2101240085723,2024-05-23 22:27:00,0,Pikri tn,NaN,Lasnamäe linnaosa,Kokkupõrge teel oleva takistusega,0.0,Tänav,Pole teada,NaN,Pole teada,Teadmine puudub,6589239.0,547834.0
30,3100220483508,2022-08-15 11:35:00,0,Muuga tee,NaN,Pirita linnaosa,Sõiduki ümberpaiskumine teel,0.0,Jalg- ja jalgrattatee,Kuiv,50.0,Selged olud,Valge aeg,6593187.0,551780.0
44,3100220535942,2022-09-06 13:00:00,0,Sõpruse pst,NaN,Kristiine linnaosa,Kokkupõrge sõidukiga küljelt,0.0,Tänav,Kuiv,50.0,Pilvised olud,Valge aeg,6586402.0,540195.0
49,3100220688966,2022-11-26 10:05:00,0,Endla tn,NaN,Kristiine linnaosa,Kokkupõrge vastutuleva sõidukiga,0.0,Tänav,Märg,50.0,Pilvised olud,Valge aeg,6588000.0,541069.0
63,3100220494754,2022-08-19 17:30:00,0,Lootsi tn,NaN,Kesklinna linnaosa,Sõiduki ümberpaiskumine teel,0.0,Tänav,Kuiv,30.0,Selged olud,Valge aeg,6589720.0,543260.0
81,2300220008025,2022-08-10 14:15:00,0,Narva mnt,NaN,Kesklinna linnaosa,Sõiduki ümberpaiskumine teel,0.0,Jalg- ja jalgrattatee,Kuiv,50.0,Pilvised olud,Valge aeg,6589214.0,543656.0


In [16]:
# add the id column into left side of table
lo_tal_21_24_clean.insert(0, 'id', range(1, len(lo_tal_21_24_clean) + 1))

In [17]:
#lo_tal_21_24_clean

For this study, it is not necessary to know the exact time of the accident, since the lighting column (light/dark time) is already available

In [18]:
# Convert to date (datetime type, but without time)
lo_tal_21_24_clean['date'] = pd.to_datetime(lo_tal_21_24_clean['date_time']).dt.date

# Check a result
lo_tal_21_24_clean[['date_time', 'date']].head(10)

,date_time,date
4,2023-04-14 12:23:00,2023-04-14
11,2022-08-22 19:21:00,2022-08-22
14,2022-06-23 19:28:00,2022-06-23
15,2023-07-23 12:55:00,2023-07-23
29,2024-05-23 22:27:00,2024-05-23
30,2022-08-15 11:35:00,2022-08-15
44,2022-09-06 13:00:00,2022-09-06
49,2022-11-26 10:05:00,2022-11-26
63,2022-08-19 17:30:00,2022-08-19
81,2022-08-10 14:15:00,2022-08-10


In [19]:
# Delete unnesessary columns
lo_tal_21_24_clean = lo_tal_21_24_clean.drop(columns=['case_id', 'date_time'])


In [20]:
# Save to csv
#lo_tal_21_24_clean.to_csv('C:/PythonGIS/geopython2025/mcarto-main/road_accs_tln_21_24.csv', 
#                         sep=';',              
#                         encoding='utf-8',  
#                         index=False)           

Now need to convert coordinates into geometry, for this needs to import the geopandas package

In [21]:
import geopandas as gpd
from shapely.geometry import Point

Usually on GIS platforms the coordinates are inverted, the x coordinate in GIS can be the y coordinate, so it is important to choose the correct coordinate when defining the geometry.

In [22]:
# Check that the column names match
# Change them if necessary to match the actual file
x_col = "y_coord"
y_col = "x_coord"

In [23]:
# Create a geometry column from points
geometry = [Point(xy) for xy in zip(lo_tal_21_24_clean[x_col], lo_tal_21_24_clean[y_col])]

In [24]:
gdf = gpd.GeoDataFrame(lo_tal_21_24_clean, geometry=geometry, crs="EPSG:3301") # 3301 is a code national coordinate system of Estonia

In [25]:
gdf.head(10)

,id,fatalities,street,ins_street,settl_unit,accs_type,pedestrian,road_type,road_cond,speed_kmh,weather,lighting,x_coord,y_coord,date,geometry
4,1,0,Peterburi tee,Kuuli tn,Lasnamäe linnaosa,Sõidukite külgkokkupõrge,0.0,Tänav,Kuiv,50.0,Selged olud,Valge aeg,6588729.0,548805.0,2023-04-14,POINT (548805 6588729)
11,2,0,Reidi tee,Narva mnt,Kesklinna linnaosa,Sõidukite külgkokkupõrge,0.0,Jalg- ja jalgrattatee,Kuiv,50.0,Selged olud,Valge aeg,6589755.0,544773.0,2022-08-22,POINT (544773 6589755)
14,3,0,Mustakivi tee,NaN,Lasnamäe linnaosa,Sõiduki ümberpaiskumine teel,0.0,Tänav,Kuiv,50.0,Selged olud,Valge aeg,6589532.0,548925.0,2022-06-23,POINT (548925 6589532)
15,4,0,Järvevana tee,NaN,Kesklinna linnaosa,Sõiduki ümberpaiskumine teel,0.0,Jalg- ja jalgrattatee,Kuiv,NaN,Pilvised olud,Valge aeg,6585478.0,542033.0,2023-07-23,POINT (542033 6585478)
29,5,0,Pikri tn,NaN,Lasnamäe linnaosa,Kokkupõrge teel oleva takistusega,0.0,Tänav,Pole teada,NaN,Pole teada,Teadmine puudub,6589239.0,547834.0,2024-05-23,POINT (547834 6589239)
30,6,0,Muuga tee,NaN,Pirita linnaosa,Sõiduki ümberpaiskumine teel,0.0,Jalg- ja jalgrattatee,Kuiv,50.0,Selged olud,Valge aeg,6593187.0,551780.0,2022-08-15,POINT (551780 6593187)
44,7,0,Sõpruse pst,NaN,Kristiine linnaosa,Kokkupõrge sõidukiga küljelt,0.0,Tänav,Kuiv,50.0,Pilvised olud,Valge aeg,6586402.0,540195.0,2022-09-06,POINT (540195 6586402)
49,8,0,Endla tn,NaN,Kristiine linnaosa,Kokkupõrge vastutuleva sõidukiga,0.0,Tänav,Märg,50.0,Pilvised olud,Valge aeg,6588000.0,541069.0,2022-11-26,POINT (541069 6588000)
63,9,0,Lootsi tn,NaN,Kesklinna linnaosa,Sõiduki ümberpaiskumine teel,0.0,Tänav,Kuiv,30.0,Selged olud,Valge aeg,6589720.0,543260.0,2022-08-19,POINT (543260 6589720)
81,10,0,Narva mnt,NaN,Kesklinna linnaosa,Sõiduki ümberpaiskumine teel,0.0,Jalg- ja jalgrattatee,Kuiv,50.0,Pilvised olud,Valge aeg,6589214.0,543656.0,2022-08-10,POINT (543656 6589214)


In [26]:
# Check column of geodataframe
print(gdf.columns.tolist())

['id', 'fatalities', 'street', 'ins_street', 'settl_unit', 'accs_type', 'pedestrian', 'road_type', 'road_cond', 'speed_kmh', 'weather', 'lighting', 'x_coord', 'y_coord', 'date', 'geometry']


P.S.
During GIS analysis it become clear that some accidents are marked in same place. In GIS it would be seen as one accident, but actually some accidents marked there. So check duplicates of coordinates.

In [27]:
duplicates = gdf.groupby(['x_coord', 'y_coord']).size()
duplicates = duplicates[duplicates > 1]

print("Coordinates with multiple accidents:")
print(duplicates)

Coordinates with multiple accidents:
x_coord    y_coord 
6581057.0  536580.0    2
6583496.0  539611.0    8
6584593.0  537930.0    2
6585537.0  535041.0    2
6585795.0  535650.0    2
6585809.0  538763.0    2
6585822.0  539985.0    4
6586064.0  541524.0    3
6586277.0  537548.0    2
           537549.0    2
6586935.0  542745.0    2
6587047.0  539486.0    2
6587192.0  536969.0    2
6587202.0  541562.0    2
6587310.0  541797.0    2
6587506.0  545094.0    2
6587642.0  542503.0    2
6587785.0  540745.0    2
6587876.0  541104.0    2
6588107.0  544081.0    2
6588423.0  542039.0    2
6588512.0  543205.0    2
6588526.0  542018.0    2
6588707.0  543742.0    2
6589083.0  541068.0    2
6589212.0  546443.0    2
6589239.0  547834.0    2
6589264.0  543222.0    3
6589402.0  543035.0    2
6589411.0  542771.0    2
6589631.0  542398.0    2
6589781.0  539901.0    2
6589863.0  542265.0    2
6589942.0  546520.0    2
6590461.0  548960.0    2
6590723.0  550628.0    2
6590905.0  539593.0    2
6591095.0  538992.

In [28]:
# Delete unnesessary columns
gdf = gdf.drop(columns=['y_coord', 'x_coord'])

In [29]:
# Save to geopackage format
# gdf.to_file("C:/PythonGIS/geopython2025/mcarto-main/road_accs_tln_21_24.gpkg")